# PEtracer lineage tracing (M2-1): s recovers tumour clones the model never saw

Control = the section's non-tumour cells, condition = malignant cells inside tumours (halo cells are
neighbour-only). Lineage (clone, subclade, fitness from the PEtracer barcode trees) is never given to the model.
Held-out clone classification (multinomial logistic, GroupKFold over 150 um spatial blocks), balanced accuracy:

| test | chance | z | s | scVI |
|---|---|---|---|---|
| T1 clone across tumours | 0.17 | 0.41 | **0.65** | 0.63 |
| T2 clone 2-1 vs 2-2 within tumour 2 | 0.50 | 0.78 | **0.80** | 0.77 |
| T3 subclade within clone 5 | 0.07 | 0.11 | **0.14** | 0.12 |
| T4 fitness within clone 5 (R2) | 0 | 0.06 | **0.18** | 0.12 |

In [ ]:
import sys, pathlib; sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), pathlib.Path.cwd() / 'notebooks'] if (p / 'repro.py').exists())))
from repro import run, require, show, fetch, fetch_fitted, STAGES, H, WK, PY, PY_VIO, GPU
# Default: download inputs + the paper's fitted Haruka outputs from Hugging Face, then analyses -> figure.
# The Haruka fit step reuses the unpacked fit; HARUKA_REFIT=1 (env) refits from scratch. Auxiliary fits (baselines, seeds) still run.
# Redraw only from saved outputs: STAGES.update(download=False, fit=False, analysis=False)  (or env HARUKA_FIGURE_ONLY=1)
# Rebuild the inputs from the raw public releases instead of Hugging Face: STAGES.update(data=True)
print(H, STAGES)

## 0 Preprocessed data (Hugging Face)

The ready-to-fit inputs are read directly from [C1vy/haruka_v2_data](https://huggingface.co/datasets/C1vy/haruka_v2_data) (cached in `hf_data/`) and linked to the paths the scripts read. The data stage below rebuilds the same files from the raw releases and is off by default.

In [ ]:
fetch('05_petracer/petracer_M2-1_lineage.h5ad', H / 'bench_petracer_lineage/m2_lineage/petracer_M2-1_lineage.h5ad')

The paper's default Haruka fit, published as fitted AnnData (input + z, s, predicted neighbourhoods; see the dataset card). It is unpacked into the fit folder, so the fit step below reuses it. Set `HARUKA_REFIT=1` before launching to refit instead.

In [ ]:
fetch_fitted('fitted/05_petracer/petracer_M2-1_lineage_haruka_fitted.h5ad', H / 'bench_petracer_lineage/m2_lineage/fits/petracer_M2-1_lineage/haruka_shared_full_niche_compgated_decoderbatch8_batchmode-decoder_only_seed0')

## 1 Data: tumour vs non-tumour design from the PEtracer release, then the lineage design

In [ ]:
run(f'{PY} bench_petracer_lineage/reaudit_2026-10-16/build_m2_tumour_vs_normal.py', 'data', log='bench_petracer_lineage/reaudit_2026-10-16/build_m2.log')
run(f'{PY} build_m2_lineage.py', 'data', cwd=H / 'bench_petracer_lineage/m2_lineage')

## 2 Haruka fit (~5 min) and the scVI baseline (~20 min)

In [ ]:
run(f'bash run_fit.sh {GPU}', 'fit', cwd=H / 'bench_petracer_lineage/m2_lineage', log='fit.log')
run(f'{PY} fit_scvi.py', 'fit', cwd=H / 'bench_petracer_lineage/m2_lineage', log='scvi.log')

## 3 Lineage tests T1-T5 (z vs s vs scVI)

In [ ]:
run(f'{PY} bench_petracer_lineage/m2_lineage/lineage_test.py', 'analysis', log='bench_petracer_lineage/m2_lineage/lineage_test.out')
import pandas as pd; pd.read_csv(H / 'bench_petracer_lineage/m2_lineage/lineage_test_summary.csv').round(3)

## 4 Panels and the page

In [ ]:
T = H / 'figures_paper/petracer'
run(f'{PY} make_petracer_figure.py', 'figure', cwd=T)
run(f'{PY} ../compose_montage.py petracer', 'figure', cwd=T)

## Figure

In [ ]:
show(H / 'figures_paper/petracer/PEtracer_full.png')